# Notebook 6 : Clustering

Notebook préparé par [Chloé-Agathe Azencott](http://cazencott.info) avec l'aide d'[Arthur Imbert](https://github.com/Henley13) et de [Victor Laigle](https://eaglev-sci.github.io/).

Dans ce notebook il s'agit d'explorer plusieurs techniques de clustering.

In [ ]:
# charger numpy as np, matplotlib as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
plt.rc('font', **{'size': 12}) # règle la taille de police globalement pour les plots (en pt)

## 1. Cercles imbriqués

Nous allons maintenant aborder un autre ensemble de méthodes non supervisées, utilisées pour regrouper des observations en fonction de leurs similitudes.

Nous commencerons par générer trois ensembles de données bidimensionnels :
- 4 taches séparées issues de distributions normales
- 2 demi-cercles imbriqués (ou « demi-lunes »)
- 2 cercles concentriques

In [ ]:
from sklearn import datasets

In [ ]:
# nombre de points
n_samples = 1000

four_blobs, four_blobs_labels = datasets.make_blobs(n_samples=n_samples, centers=4, n_features=2, random_state=170)

moons, moons_labels = datasets.make_moons(n_samples=n_samples, noise=0.05, random_state=170)

circles, circles_labels = datasets.make_circles(n_samples=n_samples, factor=.5, noise=.05, random_state=170)

Visualisons ces données :

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3))

ax[0].scatter(four_blobs[:, 0], four_blobs[:, 1], c=four_blobs_labels, s=20, alpha=0.7, cmap='viridis')
ax[1].scatter(moons[:, 0], moons[:, 1], c=moons_labels, s=20, alpha=0.7, cmap='viridis')
ax[2].scatter(circles[:, 0], circles[:, 1], c=circles_labels, s=20, alpha=0.7, cmap='viridis')

ax[0].set_title('4 blobs')
ax[1].set_title('Lunes')
ax[2].set_title('Cercles');

Supposons maintenant ne pas disposer des étiquettes. Quels algorithmes de clustering permettent de trouver deux clusters, correspondant chacun à un des cercles ?

### Algorithme des k-moyennes

L'objectif de l'algorithme k-means est retrouver $K$ clusters (et leur centroïde $\mu_k$) de manière à **minimiser la variance intra-cluster** :

\begin{align}
V = \sum_{k = 1}^{K} \sum_{x \in C_k} \frac{1}{|C_k|} (\|x - \mu_k\|^2)
\end{align}

Documentation : https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

**Implémentation manuelle**

Nous commencerons par implémenter l'algorithme « à la main », étape par étape, afin de bien comprendre et visualiser son fonctionnement. Nous verrons ensuite comment utiliser directement l'implémentation K-means dans la bibliothèque `sklearn`. Les différentes étapes de l'algorithme sont les suivantes :
1. Sélectionner le nombre `k` de clusters (hyperparamètre)
2. Initialiser les `k` centroïdes de manière aléatoire parmi nos points de données
3. Calculer les distances entre tous les points de données et ces centroïdes
4. Attribuer chaque point au cluster du centroïde le plus proche
5. Calculer la position des nouveaux centroïdes
6. Répéter les étapes 3 à 5 jusqu'à convergence, c'est-à-dire jusqu'à ce que les centroïdes ne changent plus d'une itération à l'autre

Nous allons implémenter cet algorithme sur l'ensemble de données 4 blobs, en commençant par le choix de `k` et par la sélection aléatoire de `k` points dans notre ensemble de données qui constitueront les centroïdes initiaux :

In [ ]:
np.random.seed(23)  # définir la graine. Important pour la visualisation étape par étape de k-means. Avec d'autres graines, le fonctionnement de l'algorithme n'est pas aussi clair.

k = 4
random_indices = np.random.choice(len(four_blobs), k, replace=False)
centroids_step0 = four_blobs[random_indices]

for i, centroid in enumerate(centroids_step0):
    print(f"Centroïde {i} : x = {centroid[0]},\ty = {centroid[1]}")

Examinons les données avec ces centroïdes initiaux (croix rouges).

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.scatter(four_blobs[:, 0], four_blobs[:, 1], c='grey', s=20, alpha=0.7)
ax.scatter(centroids_step0[:, 0], centroids_step0[:, 1], c='red', marker='x')

plt.show()

Nous pouvons maintenant calculer les distances entre chaque point de données et ces centroïdes.

In [ ]:
# Euclidean distance between data point and centroid
def compute_distance(data_point, centroid):
    dist = np.sqrt(np.sum((data_point - centroid)**2))
    return dist

def compute_all_distances(dataset, centroids):
    # Initialize distances array
    distances = np.zeros((k, n_samples))  # k and n_samples defined above
    
    # Calculate distance from each point to each centroid
    for i in range(k):
        for j in range(n_samples):
            distances[i, j] = compute_distance(dataset[j], centroids[i])    

    return distances

In [ ]:
distances = compute_all_distances(four_blobs, centroids_step0)

# Example of distances for the first 5 points to the k centroids
print("Distances :")
print("\t\t", " \t\t".join([f"Point {i+1}" for i in range(5)]))
for i in range(k):
    print(f"Centroïde {i}\t", "\t".join(distances[i, :5].astype(str).tolist()))

Nous attribuons désormais à chaque point le cluster correspondant au centroïde le plus proche. Pour cela, nous utilisons la fonction `argmin` de `numpy`. D'une certaine manière, cela peut être considéré comme des étiquettes prédites intermédiaires.

In [ ]:
def assign_cluster(distances):
    assignments = np.argmin(distances, axis=0)
    return assignments

intermediate_labels = assign_cluster(distances)
print(intermediate_labels[:5])  # examples of intermediate labels assigned to the data points

Nous pouvons vérifier que les étiquettes intermédiaires attribuées correspondent bien au centroïde le plus proche en comparant avec les distances calculées ci-dessus (voir cellule précédente).

Visualisons ces clusters intermédiaires et les centroïdes initiaux sur un nuage de points.

In [ ]:
def visualise_kmeans(dataset, labels, centroids, ax):

    ax.scatter(dataset[:, 0], dataset[:, 1], c=labels, s=20, alpha=0.7, cmap='viridis')
    ax.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x')
    
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
visualise_kmeans(four_blobs, intermediate_labels, centroids_step0, ax)
plt.show()


Nous devons maintenant calculer la position des nouveaux centroïdes, que nous tracerons sur un nouveau graphique.

In [ ]:
def compute_new_centroids(dataset, labels):
    centroids = np.zeros((k, dataset.shape[1]))

    for i in range(k):
        centroids[i] = dataset[labels == i].mean(axis=0)
    
    return centroids

centroids_step1 = compute_new_centroids(four_blobs, intermediate_labels)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
visualise_kmeans(four_blobs, intermediate_labels, centroids_step0, axes[0])
visualise_kmeans(four_blobs, intermediate_labels, centroids_step1, axes[1])

axes[0].set_title("Centroides initiaux")
axes[1].set_title("Centroides après une itération")

plt.show()

Nous répétons maintenant les différentes étapes :
- calcul des distances par rapport aux centroïdes,
- attribution des clusters aux points de données,
- calcul des nouveaux centroïdes.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 4))
n_iter = 10  # Nombre d'itérations de l'algorithme

current_fig = 0

centroids = centroids_step1
for i in range(n_iter):
    
    distances = compute_all_distances(four_blobs, centroids)
    intermediate_labels = assign_cluster(distances)

    # Nous affichons la visualisation toutes les 2 itérations.
    if (i+1) % 2 == 0:
        visualise_kmeans(four_blobs, intermediate_labels, centroids, axes[current_fig])
        axes[current_fig].set_title(f"Iteration {i+1}")
        current_fig += 1
        
    centroids = compute_new_centroids(four_blobs, intermediate_labels)

plt.show()


Nous voyons clairement ici comment, grâce à des itérations successives, l'algorithme est capable d'identifier correctement nos clusters, en déplaçant progressivement les centroïdes et en réajustant l'appartenance des points de données à nos clusters.

**Question** : Dans quel(s) cas l'algorithme peut-il donner de mauvais résultats ?

**Réponse** : 

L'algorithme des k-moyennes est adapté à des jeux de données dans lesquels les clusters sont de forme sphérique, convexes et de taille similaire, ce qui peut ne pas être le cas dans certaines données. Par exemple, comme on va le voir dans la suite, dans le cas des demi-cercles imbriqués ou des cercles concentriques, l'algorithme des k-moyennes risque de ne pas parvenir à identifier correctement les clusters en raison de leur forme non sphérique, et donc de donner de mauvais résultats.

Il faut aussi noter que l'algorithme est sensible à l'initialisation aléatoire des centroïdes. Si ceux-ci sont "mal" initialisés, l'algorithme peut converger vers un minimum local qui ne correspond pas à la meilleure séparation possible des clusters.

**Implémentation avec `sklearn`**

Revenons aux 3 ensembles de données créés précédemment (4 blobs, demi-lunes et cercles concentriques), et appliquons-leur l'implémentation `sklearn` de l'algorithme K-means, qui est une version optimisée des étapes que nous venons de voir.

Documentation : https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

In [ ]:
from sklearn import cluster

In [ ]:
# Instanciation des trois modèles k-means avec le
# nombre théorique de clusters (respectivement 4, 2 et 2) :
kmeans_four_blobs = cluster.KMeans(n_clusters=4)
kmeans_moons = cluster.KMeans(n_clusters=2)
kmeans_circles = cluster.KMeans(n_clusters=2)

# application aux données
kmeans_four_blobs.fit(four_blobs)
kmeans_moons.fit(moons)
kmeans_circles.fit(circles)

L'attribut `.labels_` contient, pour chaque observation, le numéro du cluster auquel cette observation est assignée.

In [ ]:
kmeans_four_blobs.labels_[0:10]  # Exemple d'étiquettes prédites pour les 10 premiers points de données

Visualisons à quoi ressemble le regroupement final pour nos trois ensembles de données :

In [ ]:
# Clustering visualization
fig, ax = plt.subplots(1, 3, figsize=(10, 3))

ax[0].scatter(four_blobs[:, 0], four_blobs[:, 1], c=kmeans_four_blobs.labels_, s=20, alpha=0.7, cmap='viridis')
ax[1].scatter(moons[:, 0], moons[:, 1], c=kmeans_moons.labels_, s=20, alpha=0.7, cmap='viridis')
ax[2].scatter(circles[:, 0], circles[:, 1], c=kmeans_circles.labels_, s=20, alpha=0.7, cmap='viridis')

ax[0].set_title('4 blobs (k=4)')
ax[1].set_title('Lunes (k=2)')
ax[2].set_title('Cercles (k=2)');

**Questions :**
- S'agit-il du regroupement attendu ? 
- Dans quel(s) cas l'algorithme k-means fonctionne-t-il correctement ?
- Pourquoi ne fonctionne-t-il pas dans les autres cas ?

**Réponses :**
- L'algorithme k-means fonctionne correctement dans le cas des 4 blobs, mais pas dans les cas des demi-lunes et des cercles concentriques. On pouvait s'y attendre puisque : 
- le k-means fonctionne bien lorsque les clusters sont de forme sphérique, convexes et de taille similaire.
- il ne fonctionne pas dans les cas des demies-lunes et des cercles concentriques car ce sont des jeux de données non-sphériques, non-convexes et que les distances entre les points et les centroïdes ne sont pas représentatives de l'appartenance à un même cluster. Un autre aspect qui aurait pu poser problème pour le k-means aurait été la présence d'outliers (ou points aberrants) dans les données, qui peuvent fausser le calcul des centroïdes.


#### Trouver K avec le coefficient de silhouette

Il arrive souvent que le nombre exact de clusters, $K$, ne soit pas connu à l'avance. Nous pouvons tout de même appliquer l'algorithme k-means et mesurer les performances du clustering afin de trouver le meilleur paramètre $K$. L'une des mesures utilisées est le **coefficient de silhouette**.

Le coefficient de silhouette (ou score) permet de **comparer les distances moyennes intra- et inter-clusters** :

\begin{align}
\text{score} = \frac{b - a}{\max(a, b)}
\end{align}

avec (pour chaque échantillon) :
- $a$ la distance moyenne intra-cluster
- $b$ la distance moyenne au cluster le plus proche

Le score est calculé pour chaque observation (avec une valeur comprise entre -1 (le pire) et 1 (le meilleur)), puis le score moyen permet d'évaluer le regroupement sur l'ensemble du nuage de points en une seule fois.

Documentation : https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html#sklearn.metrics.silhouette_score

In [ ]:
from sklearn import metrics

In [ ]:
print(f"4 blobs: Coefficient de silhouette pour k-means (k=4) : %.2f" % 
      metrics.silhouette_score(four_blobs, kmeans_four_blobs.labels_))
print(f"Lunes: Coefficient de silhouette pour  k-means (k=2) : %.2f" % 
      metrics.silhouette_score(moons, kmeans_moons.labels_))
print(f"Cercles: Coefficient de silhouette pour  k-means (k=2) : %.2f" % 
      metrics.silhouette_score(circles, kmeans_circles.labels_))


In [ ]:
k_values = range(2, 9)
names = ['4 blobs', 'Lunes', 'Cercles']
fig, ax = plt.subplots(2, 3, figsize=(16, 10))

for i, dataset in enumerate([four_blobs, moons, circles]):
    silhouettes = []
    
    for kval in k_values:

        # Initialise un modèle KMeans avec le nombre de clusters testés :
        kmeans_k = cluster.KMeans(n_clusters=kval)
        
        # Entraîne le modèle sur les données
        kmeans_k.fit(dataset)
        
        # Ajouter le score de silhouette obtenu à la liste
        silhouettes.append(metrics.silhouette_score(dataset, kmeans_k.labels_))
            
    # Visualisation du score de silhouette
    ax[0,i].plot(k_values, silhouettes)
    ax[0,i].set_xlabel("K")
    ax[0,i].set_ylabel("Coefficient de silhouette")
    ax[0,i].set_title(names[i])
    
    print("Dataset: ", names[i])
    best_silhouette = np.max(silhouettes)
    print("Coefficient de silhouette optimal: %.2f" % best_silhouette)
    best_K = k_values[silhouettes.index(best_silhouette)]
    print("Nombre correspondant de cluster K: %.0f" % best_K)
    
    # Clustering final avec le meilleur K
    kmeans_k = cluster.KMeans(n_clusters=best_K)
    kmeans_k.fit(dataset)
    ax[1,i].scatter(dataset[:, 0], dataset[:, 1], c=kmeans_k.labels_, s=20, alpha=0.7, cmap='viridis')
    ax[1,i].set_xlabel('x1')
    ax[1,i].set_ylabel('x2')
    ax[1,i].set_title('Clustering avec ' + str(best_K) + ' clusters')
fig.tight_layout()

**Conclusions :** 
- L'algorithme k-means fournit un regroupement satisfaisant pour l'ensemble de données avec quatre blobs bien séparés, même sans connaître à l'avance le nombre idéal de clusters, auquel cas le coefficient de silhouette permet de trouver ce nombre idéal.
- Cependant, malgré l'optimisation du score de silhouette, cet algorithme ne fournit pas de bons résultats pour les autres ensembles de données, qu'il s'agisse des deux lunes imbriquées ou des deux cercles concentriques.

Nous allons donc maintenant essayer un autre algorithme de regroupement et tester ses performances pour le comparer à k-means. Nous nous limiterons aux deux ensembles de données pour lesquels l'algorithme k-means ne fonctionne pas.

### DBSCAN (Clustering par densité)

L'algorithme DBSCAN (Density-Based Spatial Clustering of Applications with Noise) fonctionne en deux temps :
- Toutes les observations suffisamment proches sont connectées entre elles.
- Les observations avec un nombre minimal de voisins connectés sont considérées comme des *core samples*, à partir desquelles les clusters sont étendues. **Toutes les observations suffisamment proche d'un *core sample* appartiennent au même cluster que celui-ci**. 

Documentation : https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html

The DBSCAN algorithm takes two hyperparameters as input:
- `eps`: the *size of the neighborhood*, in other words, the distance between two data points below which one point is considered inside the neighborhood of the other.
- `min_samples`: the minimum number of neighbors for a data point to be considered a *core sample*.

In [ ]:
# Initialisation des deux modèles DBSCAN
# avec les hyperparamètres eps=0,2, min_samples=2 :
dbscan_moons = cluster.DBSCAN(eps=0.2, min_samples=2) 
dbscan_circles = cluster.DBSCAN(eps=0.2, min_samples=2) 

# Ajustement aux données
dbscan_moons.fit(moons)
dbscan_circles.fit(circles)

Une fois encore, l'attribut `.labels_` contient, pour chaque observation, le numéro du cluster auquel cette observation a été attribuée.

In [ ]:
print("Nombre d'étiquettes pour l'ensemble de données sur les lunes:", len(np.unique(dbscan_moons.labels_)))
print("Nombre d'étiquettes pour l'ensemble de données sur les cercles:", len(np.unique(dbscan_circles.labels_)))

Visualisons les clusters obtenus :

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].scatter(moons[:, 0], moons[:, 1], c=dbscan_moons.labels_, s=20, alpha=0.7, cmap='viridis')
ax[0].set_title("Clustering DBSCAN (eps=0.2)")

ax[1].scatter(circles[:, 0], circles[:, 1], c=dbscan_circles.labels_, s=20, alpha=0.7, cmap='viridis')
ax[1].set_title("Clustering DBSCAN (eps=0.2)");

Nous pouvons voir ici que l'algorithme DBSCAN est capable d'identifier les deux clusters respectifs dans les deux ensembles de données qui posaient problème à l'algorithme k-means.

Nous pouvons également noter que nous n'avons pas eu besoin de fournir au préalable un nombre de clusters pour que l'algorithme identifie correctement le nombre de clusters approprié. Cependant, l'algorithme est sensible aux deux hyperparamètres mentionnés ci-dessus, que nous allons maintenant évaluer.

#### Rôle du paramètre de taille de voisinage (`eps`)

Si `eps` est trop petit :

In [ ]:
# Choisissez des valeurs basses et élevées pour eps, par exemple 0,05 et 2,0.
eps_low = 0.05
eps_high = 2.0

# Instanciation d'un objet de clustering DBSCAN avec un eps bas et d'un autre avec un eps élevé.
dbscan_low = cluster.DBSCAN(eps=eps_low, min_samples=2)
dbscan_high = cluster.DBSCAN(eps=eps_high, min_samples=2)

# Ajuster aux données
dbscan_low.fit(circles)
dbscan_high.fit(circles)

In [ ]:
print(np.unique(dbscan_low.labels_))
print(np.unique(dbscan_high.labels_))

In [ ]:
fig = plt.figure(figsize=(5, 5))

outliers = np.where(dbscan_low.labels_ == -1)[0]
plt.scatter(circles[outliers, 0], circles[outliers, 1], marker='*', color='red', s=20)

non_outliers = np.where(dbscan_low.labels_ != -1)[0]
plt.scatter(circles[non_outliers, 0], circles[non_outliers, 1], c=dbscan_low.labels_[non_outliers], s=20, alpha=0.7, cmap='viridis')
plt.title(f"Clustering DBSCAN (eps={eps_low})");

In [ ]:
fig = plt.figure(figsize=(5, 5))
plt.scatter(circles[:, 0], circles[:, 1], c=dbscan_high.labels_, s=20, alpha=0.7, cmap='viridis')
plt.title(f"Clustering DBSCAN (eps={eps_high})");

#### Trouver eps avec le coefficient de silhouette

In [ ]:
print("Coefficient de silhouette pour DBSCAN (eps=0.2) : %.2f" % metrics.silhouette_score(circles, dbscan_circles.labels_))

In [ ]:
eps_values = np.logspace(-3, 1, 40)
silhouettes = []

for eps in eps_values:
    dbscan_eps = cluster.DBSCAN(eps=eps, min_samples=2)
    dbscan_eps.fit(circles)
    if len(np.unique(dbscan_eps.labels_)) > 1: # nécessaire pour calculer le coeff de silhouette
        silhouettes.append(metrics.silhouette_score(circles, dbscan_eps.labels_))
    else:
        silhouettes.append(-1)

In [ ]:
plt.plot(eps_values, silhouettes)
plt.xscale("log")
plt.xlabel("eps (échelle log)")
plt.ylabel("silhouette");

In [ ]:
best_silhouette = np.max(silhouettes)
print("Coefficient de silhouette optimal : %.2f" % best_silhouette)
print("Eps correspondant : %.2f" % eps_values[silhouettes.index(best_silhouette)])

**Questions :** 
- Quel est le problème ici ?
- Le coefficient de silhouette est-il adapté à notre ensemble de données ? 

**Réponses :**
- On remarque que le paramètre `eps` choisi en optimisant le coefficient de silhouette est relativement faible (0.03), et notamment plus faible qu'une valeur qu'on a justement déterminée dans les cellules précédentes comme trop faible pour obtenir un bon clustering (0.05). Il y a donc fort à parier que la valeur obtenue va également mener à un clustering avec beaucoup d'outliers et un trop grand nombre de clusters, notamment au niveau du cercle externe où la densité de points est légèrement plus faible.
- On note d'abord que malgré l'optimization, le coefficient de silhouette reste faible. Cela s'explique par le fait que le coefficient de silhouette mesure la séparation entre les clusters en se basant sur les distances moyennes intra- et inter-clusters. Avec des clusters imbriqués les uns dans les autres comme c'est le cas ici, les distances intra- et inter-clusters ne sont plus représentatives de l'appartenance à un même cluster. Le coefficient de silhouette ne peut pas capturer efficacement la structure des clusters et n'est donc pas adapté ici. 

### Index de Rand ajusté

L'index de Rand ajusté permet de **comparer un résultat de clustering avec des étiquettes**. Pour chaque paire d'observations nous regardons si elles se situent dans le même cluster ou non, dans le clustering prédit et réel. L'index prend des valeurs entre 0 (clustering aléatoire) et 1 (clustering parfait).

Documentation : https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html

__Question :__ Pourquoi ne pas utiliser une métrique d'évaluation de modèle de classification ici ?

__Réponse :__ Dans un contexte de classification supervisée, les étiquettes des classes ont une signification précise, chaque point correspond à une classe donnée et les étiquettes ne peuvent être permutées. Dans le clustering, au contraire, les étiquettes des clusters n'ont pas de signification et on peut permuter les étiquettes sans que cela n'affecte la qualité du clustering. Attribuer l'étiquette 0 au cluster A et l'étiquette 1 au cluster B ou inversement n'a pas d'importance. Les métriques de classification ne permettent pas de prendre en compte ces possibilités de permutations, et ne sont donc pas adaptées ici, d'où l'utilité de l'index de Rand ajusté.

In [ ]:
print("Index de Rand ajusté du K-means (K=2) : %.2f" % metrics.adjusted_rand_score(circles_labels, kmeans_circles.labels_))

In [ ]:
print("Index de Rand ajusté de dbscan (eps=0.2) : %.2f" % metrics.adjusted_rand_score(circles_labels, dbscan_circles.labels_))

## 2. Manchots

On reprend ici les données utilisées dans le notebook 4

In [ ]:
palmerpenguins = pd.read_csv("data/penguins.csv")

__Alternativement :__ Si vous avez besoin de télécharger le fichier (par exemple sur colab) :

In [ ]:
# !wget https://raw.githubusercontent.com/ThomasWalter/CourseFoundationsML/SIA2025/Notebooks/6-Clustering/data/penguins.csv

# palmerpenguins = pd.read_csv("penguins.csv")

In [ ]:
palmerpenguins = palmerpenguins[palmerpenguins['bill_depth_mm'].notna()]
palmerpenguins = palmerpenguins.reset_index(drop=True)

In [ ]:
penguins_X = np.array(palmerpenguins[["bill_length_mm", "body_mass_g"]])

In [ ]:
from sklearn import preprocessing

In [ ]:
# standardisation (centrer-réduire)
penguins_X = preprocessing.StandardScaler().fit_transform(penguins_X)

In [ ]:
species_names, species_int = np.unique(palmerpenguins.species, return_inverse=True)
penguins_labels = species_int
species_names

In [ ]:
plt.scatter(penguins_X[:, 0], penguins_X[:, 1], c=penguins_labels)
plt.xlabel("bill_length_mm (centrée-réduite)")
plt.ylabel("body_mass_g (centrée-réduite)")

C'est maintenant à vous de tester différents algorithmes de clustering sur ces données et d'évaluer leurs performances. Réussirez-vous à obtenir un clustering parfait ?

En plus des algorithmes de clustering utilisés ci-dessus, vous pouvez essayer les mélanges gaussiens ([GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture. GaussianMixture.html)), par exemple, ou d'autres méthodes issues des modules `cluster` ou `mixture`. Pour chacune d'entre elles, fournissez les coefficients de silhouette et les indices de Rand ajustés que vous obtenez, ainsi qu'une visualisation du clustering obtenu.

### KMeans

In [ ]:
# initialisation d'un k-means avec k=3
kmeans = cluster.KMeans(n_clusters=3)

# application aux données 
kmeans.fit(penguins_X)

In [ ]:
#plt.scatter(penguins_X[:, 0], penguins_X[:, 1], c=penguins_labels, marker='o')
plt.scatter(penguins_X[:, 0], penguins_X[:, 1], c=kmeans.labels_, marker='*')


plt.xlabel("bill_length_mm (centrée-réduite)")
plt.ylabel("body_mass_g (centrée-réduite)")

In [ ]:
print("Coefficient de silhouette pour le k-means (k=3) : %.2f" % metrics.silhouette_score(penguins_X, kmeans.labels_))

In [ ]:
print("Index de Rand ajusté du K-means (K=3) : %.2f" % metrics.adjusted_rand_score(penguins_labels, kmeans.labels_))

### DBSCAN

In [ ]:
eps_values = np.logspace(-3, 1, 40)
silhouettes = []

for eps in eps_values:
    dbscan_eps = cluster.DBSCAN(eps=eps, min_samples=2)
    dbscan_eps.fit(penguins_X)
    if len(np.unique(dbscan_eps.labels_)) > 1: # nécessaire pour calculer le coeff de silhouette
        silhouettes.append(metrics.silhouette_score(penguins_X, dbscan_eps.labels_))
    else:
        silhouettes.append(-1)

In [ ]:
plt.plot(eps_values, silhouettes)
plt.xscale("log")
plt.xlabel("eps (échelle log)")
plt.ylabel("silhouette")

In [ ]:
best_silhouette = np.max(silhouettes)
print("Coefficient de silhouette optimal : %.2f" % best_silhouette)
best_eps = eps_values[silhouettes.index(best_silhouette)]
print("Eps correspondant : %.2f" % best_eps)

In [ ]:
dbscan_opt = cluster.DBSCAN(eps=best_eps, min_samples=2)
dbscan_opt.fit(penguins_X)

In [ ]:
np.unique(dbscan_opt.labels_)

In [ ]:
print("Index de Rand ajusté de DBSCAN : %.2f" % metrics.adjusted_rand_score(penguins_labels, dbscan_opt.labels_))

In [ ]:
#plt.scatter(penguins_X[:, 0], penguins_X[:, 1], c=penguins_labels, marker='o')
plt.scatter(penguins_X[:, 0], penguins_X[:, 1], c=dbscan_opt.labels_, marker='*')


plt.xlabel("bill_length_mm (centrée-réduite)")
plt.ylabel("body_mass_g (centrée-réduite)")

### Modèle de mélange gaussien 

Le modèle de mélange de gaussiennes cherche à **optimiser les paramètres d'un nombre fini de gaussiennes** aux données. 

Documentation : https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html

In [ ]:
from sklearn import mixture

In [ ]:
# initialisation d'un k-means avec k=3
gmm = mixture.GaussianMixture(n_components=3)

# application aux données 
gmm.fit(penguins_X)

# prédiction des clusters
gmm_labels = gmm.predict(penguins_X)

In [ ]:
#plt.scatter(penguins_X[:, 0], penguins_X[:, 1], c=penguins_labels, marker='o')
plt.scatter(penguins_X[:, 0], penguins_X[:, 1], c=gmm_labels, marker='*')


plt.xlabel("bill_length_mm (centrée-réduite)")
plt.ylabel("body_mass_g (centrée-réduite)")

In [ ]:
print("Coefficient de silhouette pour le GMM (k=3) : %.2f" % metrics.silhouette_score(penguins_X, gmm_labels))

In [ ]:
print("Index de Rand ajusté du GMM (K=3) : %.2f" % metrics.adjusted_rand_score(penguins_labels, gmm_labels))

## Conclusion

Nous sommes arrivés à la fin de ce notebook. Voici un résumé de ce que nous avons couvert, avec les points clés :

- Nous avons exploré plusieurs méthodes de **clustering** (regroupement non supervisé) sur jeux synthétiques et sur les données des manchots (`palmerpenguins`).

- Nous avons implémenté et visualisé l'algorithme **k-means** (manuel puis via `sklearn`) : il fonctionne bien pour des blobs sphériques bien séparés, mais échoue sur des structures non-convexes (lunas, cercles concentriques).

- Nous avons utilisé le **coefficient de silhouette** pour estimer un bon nombre de clusters K et évaluer la qualité du regroupement.

- Nous avons testé **DBSCAN** (clustering par densité) qui détecte correctement les lunes et les cercles sans fournir K, mais reste sensible aux hyperparamètres `eps` et `min_samples`.

- Nous avons comparé les approches à l'aide de métriques adaptées au clustering : **coefficient de silhouette** et **index de Rand ajusté** (comparaison à des étiquettes de référence quand disponibles).

- Sur les données réelles (manchots) nous avons essayé plusieurs méthodes : **k-means**, **DBSCAN** et **Gaussian Mixture Models (GMM)**, en mesurant silhouette et adjusted Rand pour comparer les performances.

- Points pratiques : standardiser les variables avant clustering, visualiser les regroupements, et tester plusieurs méthodes/hyperparamètres — il n'existe pas d'algorithme universel, le bon choix dépend de la structure des données.
